# Fix rapido: baseline.py v1.1

**Bug:** usaba `torch.relu(x, inplace=False)` pero `torch.relu` no acepta `inplace`.

**Fix:** cambiar a `torch.nn.functional.relu(x, inplace=False)`.

Este notebook sobrescribe solo `baseline.py` y vuelve a correr los smoke tests
del notebook anterior (celdas 4, 5, 6).


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR = f'{PROJECT_ROOT}/code'

import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)


In [ ]:
!pip install -q torchxrayvision scipy


## 2. Sobrescribir `baseline.py`

In [ ]:
from pathlib import Path

content = r'''"""
src/models/baseline.py  (v1.1 - fix de relu)

Modelo baseline del Sprint 1: DenseNet121 preentrenado en NIH ChestX-ray14
(via torchxrayvision) adaptado a clasificacion binaria.

FIX v1.1: cambio `torch.relu(x, inplace=False)` por `torch.relu(x)`.
`torch.relu` no acepta `inplace` (solo `torch.nn.functional.relu` lo hace).
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchxrayvision as xrv


class BaselineCNN(nn.Module):
    """
    DenseNet121 de torchxrayvision con cabeza de clasificacion binaria.

    Args:
        backbone_weights: string del modelo preentrenado en xrv
                          (default: "densenet121-res224-nih").
        num_classes: numero de clases de salida (2 para binaria).
        finetune_mode: "head" (solo capa final) o "full" (todo el modelo).
    """

    def __init__(self, backbone_weights: str = "densenet121-res224-nih",
                 num_classes: int = 2, finetune_mode: str = "head"):
        super().__init__()

        if finetune_mode not in {"head", "full"}:
            raise ValueError(f"finetune_mode debe ser 'head' o 'full', "
                             f"no '{finetune_mode}'")
        self.finetune_mode = finetune_mode
        self.num_classes = num_classes

        # Cargar DenseNet121 preentrenado de xrv
        self.backbone = xrv.models.DenseNet(weights=backbone_weights)

        # Dim de features (1024 para DenseNet121)
        num_features = self.backbone.classifier.in_features

        # Reemplazar classifier (14 clases multi-label) por uno binario
        self.backbone.classifier = nn.Linear(num_features, num_classes)

        self._apply_finetune_mode()

    def _apply_finetune_mode(self) -> None:
        if self.finetune_mode == "head":
            for param in self.backbone.parameters():
                param.requires_grad = False
            for param in self.backbone.classifier.parameters():
                param.requires_grad = True
        else:  # full
            for param in self.backbone.parameters():
                param.requires_grad = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass. Evitamos la sigmoid interna de xrv llamando los pasos
        manualmente y devolvemos logits crudos.
        """
        features = self.backbone.features(x)
        out = F.relu(features, inplace=False)  # <-- FIX: usar F.relu, no torch.relu
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        logits = self.backbone.classifier(out)
        return logits

    def trainable_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def total_params(self) -> int:
        return sum(p.numel() for p in self.parameters())


# ---------------------------------------------------------------------------
# Factory
# ---------------------------------------------------------------------------
def build_baseline_model(cfg) -> BaselineCNN:
    return BaselineCNN(
        backbone_weights=cfg.model['backbone'],
        num_classes=cfg.model['num_classes'],
        finetune_mode=cfg.model['finetune_mode'],
    )


# ---------------------------------------------------------------------------
# Smoke test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import sys
    from pathlib import Path

    code_dir = Path(__file__).resolve().parent.parent.parent
    if str(code_dir) not in sys.path:
        sys.path.insert(0, str(code_dir))

    from src.training.utils import load_config, get_device

    cfg = load_config(code_dir / "configs" / "baseline.yaml")
    device = get_device()

    model = build_baseline_model(cfg).to(device)
    print(f"backbone: {cfg.model['backbone']}")
    print(f"num_classes: {cfg.model['num_classes']}")
    print(f"finetune_mode: {cfg.model['finetune_mode']}")
    print(f"parametros totales:     {model.total_params():>12,}")
    print(f"parametros entrenables: {model.trainable_params():>12,}")

    x = torch.randn(4, 1, 224, 224, device=device) * 500
    logits = model(x)
    print(f"\nForward OK: input={x.shape} -> output={logits.shape}")
    probs = torch.softmax(logits, dim=1)
    print(f"probs[0]: {probs[0].tolist()}")
'''
path = f'{CODE_DIR}/src/models/baseline.py'
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


## 3. Smoke test del modelo corregido

In [ ]:
# Limpiar cache de imports
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'):
        del sys.modules[mod]

import torch
from src.training.utils import load_config, get_device
from src.models.baseline import build_baseline_model

cfg = load_config(f'{CODE_DIR}/configs/baseline.yaml')
device = get_device()

model = build_baseline_model(cfg).to(device)
model.eval()

print(f"Modo: {cfg.model['finetune_mode']}")
print(f'Parametros totales:     {model.total_params():>12,}')
print(f'Parametros entrenables: {model.trainable_params():>12,}')

# Forward dummy - AHORA DEBE FUNCIONAR
x = torch.randn(4, 1, 224, 224, device=device) * 500
with torch.no_grad():
    logits = model(x)
print(f'\n✓ Forward OK:')
print(f'  input shape:  {x.shape}')
print(f'  output shape: {logits.shape}')
print(f'  logits range: [{logits.min():.3f}, {logits.max():.3f}]')

probs = torch.softmax(logits, dim=1)
print(f'  probs[0]: {probs[0].tolist()}  (suma = {probs[0].sum().item():.3f})')


## 4. Evaluacion del modelo sin entrenar sobre val_ds

In [ ]:
from torch.utils.data import DataLoader
from src.datasets.nih import build_nih_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.evaluation.metrics import compute_metrics, logits_to_probs

train_tfm = build_train_transform(image_size=cfg.preprocessing['image_size'])
eval_tfm = build_eval_transform(image_size=cfg.preprocessing['image_size'])

_, val_ds, _ = build_nih_datasets(cfg, train_tfm, eval_tfm)

val_loader = DataLoader(val_ds, batch_size=cfg.training['batch_size'],
                        shuffle=False, num_workers=0)

all_probs, all_labels = [], []
model.eval()
with torch.no_grad():
    for batch in val_loader:
        logits = model(batch['image'].to(device))
        probs = logits_to_probs(logits, positive_class=1)
        all_probs.append(probs.cpu())
        all_labels.append(batch['label'])

import torch as _torch
all_probs = _torch.cat(all_probs).numpy()
all_labels = _torch.cat(all_labels).numpy()

metrics_init = compute_metrics(all_labels, all_probs, threshold=0.5)
print(f'Muestras en val: {len(all_labels)} | positivos: {int(all_labels.sum())}')
print()
print('Metricas del modelo SIN entrenar (classifier head con pesos random):')
print(metrics_init)
print()
print('Nota: estas metricas son la LINEA DE PARTIDA (cercanas al azar).')
print('Despues de entrenar esperamos AUC > 0.70.')


## 5. Siguiente paso

Si todo corrio sin errores, el pipeline esta cerrado hasta evaluacion.
Vuelve al chat para recibir el loop de entrenamiento:

1. `src/training/train.py`
2. `scripts/run_train.py`

Eso es lo ultimo antes del primer entrenamiento real.
